<a href="https://colab.research.google.com/github/DifferentiableUniverseInitiative/JaxPM/blob/main/docs/notebooks/01-Introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to JaxPM

A minimal end-to-end example: build Gaussian initial conditions from a linear matter
power spectrum, displace particles with first-order LPT, evolve them with a symplectic
Particle-Mesh step (diffrax `SemiImplicitEuler`), and `paint` the density field.

In [ ]:
# Install JaxPM. Runs only when the package is missing (e.g. on Colab);
# an existing local (editable) install is left untouched.
try:
    import jaxpm  # noqa: F401
except ImportError:
    %pip install -q --upgrade jax
    %pip install -q jaxpm

In [ ]:
import jax
import jax.numpy as jnp
import jax_cosmo as jc

from diffrax import (ODETerm, SaveAt, SemiImplicitEuler, ConstantStepSize,
                     diffeqsolve)

from jaxpm.ode import symplectic_ode
from jaxpm.painting import paint
from jaxpm.pm import linear_field, lpt

In [ ]:
mesh_shape = [128, 128, 128]
box_size = [128., 128., 128.]
snapshots = jnp.array([0.1, 0.5, 1.0])


@jax.jit
def run_simulation(omega_c, sigma8):
    cosmo = jc.Planck15(Omega_c=omega_c, sigma8=sigma8)

    # Linear matter power spectrum -> Gaussian initial conditions.
    k = jnp.logspace(-4, 1, 128)
    pk = jc.power.linear_matter_power(cosmo, k)
    pk_fn = lambda x: jnp.interp(x.reshape([-1]), k, pk).reshape(x.shape)
    initial_conditions = linear_field(mesh_shape, box_size, pk_fn,
                                      seed=jax.random.PRNGKey(0))

    # First-order LPT displacements from a uniform particle grid (particles=None
    # -> the field is stored as displacements dx, not absolute positions).
    dx, p, _ = lpt(cosmo, initial_conditions, a=0.1, order=1)

    # Symplectic Particle-Mesh step: drift/kick operators integrated with the
    # diffrax SemiImplicitEuler (symplectic Euler) solver.
    drift, kick = symplectic_ode(mesh_shape, cosmo,
                                 initial_particles='uniform',
                                 deconvolution=True)
    sol = diffeqsolve((ODETerm(kick), ODETerm(drift)), SemiImplicitEuler(),
                      t0=0.1, t1=1.0, dt0=0.05, y0=(p, dx), args=cosmo,
                      saveat=SaveAt(ts=snapshots),
                      stepsize_controller=ConstantStepSize())

    # sol.ys = (momenta, displacements) sampled at each requested scale factor.
    return initial_conditions, dx, sol.ys[1]

In [ ]:
initial_conditions, lpt_dx, ode_dx = run_simulation(0.25, 0.8)

In [ ]:
from jaxpm.plotting import plot_fields_single_projection

# The evolved state is a displacement field, so paint it with
# initial_particles='uniform' (a uniform grid is added internally).
density = lambda dx: jnp.log10(
    paint(dx, initial_particles='uniform', order="cic") + 1)

fields = {"Initial Conditions": initial_conditions, "LPT Field": density(lpt_dx)}
for a, dx_a in zip(snapshots[1:], ode_dx[1:]):
    fields[f"a = {float(a):.1f}"] = density(dx_a)
plot_fields_single_projection(fields)